# Upload and Verify the Recommendation Model

The trained recommendation model has already been created in the training notebook and saved as `recommendation_model.pkl`.

This deployment notebook does not perform any model training. The previously generated `.pkl` file is uploaded so that the standalone Streamlit application can be developed and tested independently of the training notebook.

The model package should contain all artifacts required for inference:
- the trained NearestNeighbors model
- the hotel feature matrix
- the hotel catalogue
- the cleaned historical user–hotel interaction data

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving recommendation_model.pkl to recommendation_model.pkl


In [ ]:
import os

model_path = "recommendation_model.pkl"

print(
    "Model exists:",
    os.path.exists(model_path)
)

if os.path.exists(model_path):
    print(
        "Model size:",
        round(
            os.path.getsize(model_path) / 1024,
            2
        ),
        "KB"
    )

Model exists: True
Model size: 1828.56 KB


# Load and Inspect the Deployment Package

The saved recommendation package is loaded using Joblib.

The purpose of this step is to confirm that the deployment file contains everything required by the Streamlit application.

No training or preprocessing is performed here. The objects are loaded exactly as they were saved by the training notebook.

In [ ]:
import joblib

deployment_package = joblib.load(
    "recommendation_model.pkl"
)

print(
    "Deployment package loaded successfully."
)

print("\nPackage contents:")

for key, value in deployment_package.items():
    print(
        f"{key}: {type(value)}"
    )

Deployment package loaded successfully.

Package contents:
recommendation_model: <class 'sklearn.neighbors._unsupervised.NearestNeighbors'>
hotel_feature_matrix: <class 'scipy.sparse._csr.csr_matrix'>
hotel_catalog: <class 'pandas.core.frame.DataFrame'>
hotels_clean: <class 'pandas.core.frame.DataFrame'>


# Verify the Saved Artifacts

The dimensions of the saved objects are checked before creating the deployment application.

This confirms that the model, hotel feature representation, hotel catalogue, and historical interaction data are consistent with the results produced in the training notebook.

In [ ]:
recommendation_model = deployment_package[
    "recommendation_model"
]

hotel_feature_matrix = deployment_package[
    "hotel_feature_matrix"
]

hotel_catalog = deployment_package[
    "hotel_catalog"
]

hotels_clean = deployment_package[
    "hotels_clean"
]

print(
    "Recommendation model:",
    type(recommendation_model)
)

print(
    "Feature matrix shape:",
    hotel_feature_matrix.shape
)

print(
    "Hotel catalogue shape:",
    hotel_catalog.shape
)

print(
    "Historical data shape:",
    hotels_clean.shape
)

Recommendation model: <class 'sklearn.neighbors._unsupervised.NearestNeighbors'>
Feature matrix shape: (9, 19)
Hotel catalogue shape: (9, 3)
Historical data shape: (40552, 8)


# Test Recommendation Inference

The saved recommendation model is tested independently of the training notebook.

A hotel is selected from the saved hotel catalogue and the nearest hotels are retrieved using the saved feature matrix and NearestNeighbors model.

This verifies that the serialized model package can perform inference without requiring the original training variables.

In [ ]:
test_hotel = hotel_catalog["name"].iloc[0]

hotel_index = hotel_catalog.index[
    hotel_catalog["name"] == test_hotel
][0]

n_neighbors = min(
    6,
    len(hotel_catalog)
)

distances, indices = (
    recommendation_model.kneighbors(
        hotel_feature_matrix[hotel_index],
        n_neighbors=n_neighbors
    )
)

similar_hotels = hotel_catalog.iloc[
    indices[0][1:]
].copy()

similar_hotels[
    "similarity_score"
] = 1 - distances[0][1:]

print(
    "Test hotel:",
    test_hotel
)

print("\nSimilar hotels:")

display(
    similar_hotels
)

Test hotel: Hotel A

Similar hotels:


,name,place,price,similarity_score
4,Hotel AU,Recife (PE),312.83,0.427385
1,Hotel K,Salvador (BH),263.41,0.252186
8,Hotel BP,Brasilia (DF),247.62,0.173785
2,Hotel BD,Natal (RN),242.88,0.148360
3,Hotel Z,Aracaju (SE),208.04,-0.052812


# Prepare the Streamlit Environment

Streamlit is installed in the deployment testing environment.

The Streamlit application will be a separate Python file and will not contain any model training code.

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 95.5 MB/s eta 0:00:00


In [ ]:
import streamlit

print(
    "Streamlit version:",
    streamlit.__version__
)

Streamlit version: 1.62.0


# Create the Standalone Streamlit Application

The Streamlit application is created as a separate `app.py` file.

The application loads `recommendation_model.pkl` at runtime and uses the saved model package to generate recommendations.

No model training, feature fitting, or evaluation is performed by the application.

The application provides:
- user selection
- travel history
- number of recommendations
- personalized hotel recommendations
- hotel destination
- hotel price
- recommendation score

In [28]:
app_code = r'''
import streamlit as st
import pandas as pd
import joblib


st.set_page_config(
    page_title="Voyage Analytics",
    page_icon="✈️",
    layout="wide"
)


@st.cache_resource
def load_package():

    return joblib.load(
        "recommendation_model.pkl"
    )


package = load_package()

recommendation_model = package[
    "recommendation_model"
]

hotel_feature_matrix = package[
    "hotel_feature_matrix"
]

hotel_catalog = package[
    "hotel_catalog"
]

hotels_clean = package[
    "hotels_clean"
]


def recommend_similar_hotels(
    hotel_name,
    top_n=5
):

    matching_indices = hotel_catalog.index[
        hotel_catalog["name"] == hotel_name
    ].tolist()

    if not matching_indices:
        return pd.DataFrame()

    hotel_index = matching_indices[0]

    n_neighbors = min(
        top_n + 1,
        len(hotel_catalog)
    )

    distances, indices = (
        recommendation_model.kneighbors(
            hotel_feature_matrix[hotel_index],
            n_neighbors=n_neighbors
        )
    )

    recommendations = hotel_catalog.iloc[
        indices[0][1:]
    ].copy()

    recommendations[
        "similarity_score"
    ] = 1 - distances[0][1:]

    return recommendations.reset_index(
        drop=True
    )


def recommend_for_user(
    user_code,
    top_n=5
):

    user_history = hotels_clean[
        hotels_clean["usercode"] == user_code
    ]

    if user_history.empty:
        return pd.DataFrame()

    previous_hotels = (
        user_history["name"]
        .dropna()
        .unique()
        .tolist()
    )

    candidate_scores = {}

    for hotel_name in previous_hotels:

        if hotel_name not in set(
            hotel_catalog["name"]
        ):
            continue

        n_to_search = min(
            top_n * 3,
            max(
                0,
                len(hotel_catalog) - 1
            )
        )

        if n_to_search == 0:
            continue

        similar_hotels = recommend_similar_hotels(
            hotel_name,
            top_n=n_to_search
        )

        for _, row in similar_hotels.iterrows():

            candidate = row["name"]
            score = row["similarity_score"]

            if candidate in previous_hotels:
                continue

            candidate_scores[candidate] = (
                candidate_scores.get(
                    candidate,
                    0
                ) + score
            )

    if not candidate_scores:
        return pd.DataFrame()

    recommendations = pd.DataFrame(
        candidate_scores.items(),
        columns=[
            "name",
            "recommendation_score"
        ]
    )

    recommendations = (
        recommendations
        .sort_values(
            "recommendation_score",
            ascending=False
        )
        .head(top_n)
    )

    recommendations = recommendations.merge(
        hotel_catalog,
        on="name",
        how="left"
    )

    return recommendations.reset_index(
        drop=True
    )


st.title("✈️ Voyage Analytics")

st.subheader(
    "Personalized Hotel Recommendation System"
)

st.write(
    "Select a user to view their travel history "
    "and receive personalized hotel recommendations."
)


user_ids = sorted(
    hotels_clean["usercode"]
    .dropna()
    .unique()
    .tolist()
)

selected_user = st.selectbox(
    "Select User",
    user_ids
)


user_history = hotels_clean[
    hotels_clean["usercode"] == selected_user
].copy()


if not user_history.empty:

    st.subheader("Travel History")

    history_columns = [
        "name",
        "place",
        "price",
        "days",
        "date"
    ]

    available_columns = [
        column
        for column in history_columns
        if column in user_history.columns
    ]

    st.dataframe(
        user_history[
            available_columns
        ],
        use_container_width=True
    )

    st.subheader(
        "Recommended Hotels"
    )

    top_n = st.slider(
        "Number of recommendations",
        min_value=1,
        max_value=5,
        value=5
    )

    recommendations = recommend_for_user(
        selected_user,
        top_n=top_n
    )

    if recommendations.empty:

        st.warning(
            "No new hotels are available for "
            "recommendation for this user."
        )

    else:

        display_columns = [
            "name",
            "place",
            "price",
            "recommendation_score"
        ]

        available_display_columns = [
            column
            for column in display_columns
            if column in recommendations.columns
        ]

        st.dataframe(
            recommendations[
                available_display_columns
            ],
            use_container_width=True
        )

else:

    st.warning(
        "No historical hotel data found for "
        "the selected user."
    )
'''

with open(
    "app.py",
    "w"
) as file:
    file.write(app_code)

print("app.py created successfully.")

app.py created successfully.


# Verify the Standalone Application File

The generated `app.py` file is checked to ensure that it exists and contains the expected deployment code.

The application is kept separate from the training notebook and will be the file used by the deployment environment.

In [29]:
import os

app_path = "/content/app.py"

print(
    "app.py exists:",
    os.path.exists(app_path)
)

if os.path.exists(app_path):
    print(
        "File size:",
        round(
            os.path.getsize(app_path) / 1024,
            2
        ),
        "KB"
    )

app.py exists: True
File size: 4.76 KB


In [30]:
from google.colab import files

files.download(
    "/content/app.py"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Run the Standalone Streamlit Application

The standalone `app.py` is launched using the saved `recommendation_model.pkl`.

The application should work without executing the training notebook because all required recommendation artifacts are loaded from the serialized model package.

In [ ]:
!streamlit run app.py \
    --server.address=0.0.0.0 \
    --server.port=8501 \
    > /content/streamlit.log 2>&1 &

In [ ]:
import time

time.sleep(5)

print(
    open("/content/streamlit.log").read()
)



2026-08-24 11:07:16.185 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.186.182.100:8501




## Verify the Streamlit Server

Before exposing the Streamlit application to the browser, the server process is checked directly from the Colab runtime.

This distinguishes an application startup problem from a networking or port-access problem.

In [ ]:
import os
import time

time.sleep(3)

print(
    open("/content/streamlit.log").read()
)

print("\nRunning Streamlit processes:")

os.system(
    "ps aux | grep '[s]treamlit'"
)



2026-08-24 11:07:16.185 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.186.182.100:8501



Running Streamlit processes:


0

## Install Cloudflare Tunnel

Cloudflare Tunnel is installed in the Colab deployment environment so that the locally running Streamlit application can be accessed through a temporary HTTPS URL.

The tunnel is used only for testing the Streamlit interface. It is not part of the recommendation model or the final `app.py`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared

!chmod +x /content/cloudflared

print("Cloudflare Tunnel installed.")

Cloudflare Tunnel installed.


In [ ]:
import os

print(
    "cloudflared exists:",
    os.path.exists("/content/cloudflared")
)

print(
    "Executable:",
    os.access(
        "/content/cloudflared",
        os.X_OK
    )
)

cloudflared exists: True
Executable: True


## Start the Cloudflare Tunnel

The Cloudflare Quick Tunnel forwards the Streamlit service running on port 8501 to a temporary HTTPS URL.

The generated URL will be used only to verify the deployed Streamlit application in a browser.

In [ ]:
!/content/cloudflared tunnel --url http://localhost:8501